# Informed Search

### Maze Environments
The environments used is **SmallMaze** (visible in the figure).

<img src="images/maze.png" width="300">

The agent starts in cell $(0, 2)$ and has to reach the treasure in $(4, 3)$.

In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('../tools'))
if module_path not in sys.path:
    sys.path.append(module_path)

from utils.ai_lab_functions import *
import gym
import envs

### Uniform-Cost Search (UCS)

The cost of performing an action is supposed to be always 1

In [2]:
def present_with_higher_cost(queue, node):
    if node.state in queue:
        if queue[node.state].pathcost > node.pathcost: 
            return True
    return False

In [3]:
def ucs(environment):
    queue = PriorityQueue()
    queue.add(Node(environment.startstate))
    
    explored = set()
    time_cost = 1
    space_cost = 1
    
    while True:
        if queue.is_empty(): 
            return None, time_cost, space_cost 
        
        # Retrieve node from the queue
        node = queue.remove()  
        if node.state == environment.goalstate: 
            return build_path(node), time_cost, space_cost
        
        explored.add(node.state)
        
        # Look around
        for action in range(environment.action_space.n):
            
            # Child node where value and pathcost are both the pathcost of parent + 1
            child = Node(environment.sample(node.state, action), node, node.pathcost + 1, node.pathcost + 1)  
            time_cost += 1
            
            if child.state not in queue and child.state not in explored:
                queue.add(child)
                
            elif present_with_higher_cost(queue, child):
                queue.replace(child)
                
        space_cost = max(space_cost, len(queue) + len(explored))

### Check the results

In [4]:
# Create and render the environment
env = gym.make("SmallMaze-v0")
env.render()
solution, time, memory = ucs(env)

CheckResult_UCS(solution, time, memory, env)

[['C' 'C' 'S' 'C']
 ['C' 'C' 'W' 'C']
 ['C' 'C' 'C' 'C']
 ['C' 'W' 'W' 'W']
 ['C' 'C' 'C' 'G']]

##########################################
#####  UNIFORM GRAPH SEARCH PROBLEM  #####
##########################################
Solution: [(np.int64(0), np.int64(1)), (np.int64(0), np.int64(0)), (np.int64(1), np.int64(0)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 61
Max n° of nodes in memory: 16


### Distance Heuristics

Informed search requires a distance heuristic to estimate the distance between a state and the goal:

- *l1_norm(p1, p2)* - Computes the L1 norm (also known as the manhattan distance) between two points specified as tuples of coordinates.
- *l2_norm(p1, p2)* - Computes the L2 norm between two points specified as tuples of coordinates.
- *chebyshev(p1, p2)* - Computes the Chebyshev distance between two points specified as tuples of coordinates. Similar to the L1 norm but diagonal moves are also considered.

In [5]:
p1 = (0, 2)
p2 = (4, 0)
print("L1 norm heuristic value: {}".format(Heu.l1_norm(p1, p2)))
print("L2 norm heuristic value: {}".format(Heu.l2_norm(p1, p2)))
print("Chebyshev heuristic value: {}".format(Heu.chebyshev(p1, p2)))

L1 norm heuristic value: 6
L2 norm heuristic value: 4.47213595499958
Chebyshev heuristic value: 4


In [6]:
def present_with_higher_value(queue, node):
    if node.state in queue:
        if queue[node.state].value > node.value: 
            return True
    return False

### Greedy Tree Search

In [7]:
def greedy_tree_search(environment, timeout=10000):
    goalpos = environment.state_to_pos(environment.goalstate)
    queue = PriorityQueue()
    queue.add(Node(environment.startstate))
    
    time_cost = 1
    space_cost = 1
    
    while True:
        # timeout check
        if time_cost >= timeout:
            return ("time-out", time_cost, space_cost)
        
        if queue.is_empty(): 
            return None, time_cost, space_cost 
        
        # Retrieve node from the queue
        node = queue.remove()  
        if node.state == environment.goalstate: 
            return build_path(node), time_cost, space_cost
                
        for action in range(environment.action_space.n):
            child = Node(
                environment.sample(node.state, action),
                node,
                node.pathcost + 1,
                #l1_norm / l2_norm / chebyshev
                Heu.l1_norm(environment.state_to_pos(environment.sample(node.state, action)), goalpos))  
            time_cost += 1
            
            queue.add(child)
                
        space_cost = max(space_cost, len(queue))

### Greedy Graph Search

In [8]:
def greedy_graph_search(environment):
    """
    Greedy-best-first Graph search
    
    Args:
        problem: OpenAI Gym environment
        
    Returns:
        (path, time_cost, space_cost): solution as a path and stats.
    """
    goalpos = environment.state_to_pos(environment.goalstate)
    queue = PriorityQueue()
    queue.add(Node(environment.startstate))
    
    explored = set()
    time_cost = 1
    space_cost = 1
    
    while True:
        if queue.is_empty(): 
            return None, time_cost, space_cost 
        
        # Retrieve node from the queue
        node = queue.remove()  
        if node.state == environment.goalstate: 
            return build_path(node), time_cost, space_cost
        
        explored.add(node.state)
        
        for action in range(environment.action_space.n):
            child = Node(
                environment.sample(node.state, action),
                node,
                node.pathcost + 1,
                #l1_norm / l2_norm / chebyshev
                Heu.l1_norm(environment.state_to_pos(environment.sample(node.state, action)), goalpos))  
            time_cost += 1
            
            if child.state not in queue and child.state not in explored:
                queue.add(child)
                
            elif present_with_higher_value(queue, child):
                queue.replace(child)
                
        space_cost = max(space_cost, len(queue) + len(explored))

### Check the results

In [9]:
def greedy(environment, search_type):
    path, time_cost, space_cost = search_type(environment)
    return path, time_cost, space_cost

# envname = "SmallMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts = greedy(environment, greedy_tree_search)
solution_gs, time_gs, memory_gs = greedy(environment, greedy_graph_search)
heuristic = str(input('Which heuristic did you use? (l1_norm, l2_norm, chebyshev)\n'))

results = CheckResult_L2A1([solution_ts, time_ts, memory_ts], [solution_gs, time_gs, memory_gs], heuristic, env)
results.check_sol_ts()
results.check_sol_gs()

### A* Tree Search

In [11]:
def astar_tree_search(environment):
    goalpos = environment.state_to_pos(environment.goalstate)
    queue = PriorityQueue()
    queue.add(Node(environment.startstate))
    
    time_cost = 1
    space_cost = 1
    
    while True:
        if queue.is_empty(): 
            return None, time_cost, space_cost 
        
        # Retrieve node from the queue
        node = queue.remove()  
        if node.state == environment.goalstate: 
            return build_path(node), time_cost, space_cost
                
        for action in range(environment.action_space.n):
            child = Node(
                environment.sample(node.state, action),
                node,
                node.pathcost + 1,
                #l1_norm / l2_norm / chebyshev
                node.pathcost + 1 + Heu.l1_norm(
                    environment.state_to_pos(environment.sample(node.state, action)),
                    goalpos))  
            time_cost += 1
            
            queue.add(child)
                
        space_cost = max(space_cost, len(queue))

### A* Graph Search

In [12]:
def astar_graph_search(environment):
    goalpos = environment.state_to_pos(environment.goalstate)
    queue = PriorityQueue()
    queue.add(Node(environment.startstate))
    
    explored = set()
    time_cost = 1
    space_cost = 1
    
    while True:
        if queue.is_empty(): 
            return None, time_cost, space_cost 
        
        # Retrieve node from the queue
        node = queue.remove()  
        if node.state == environment.goalstate: 
            return build_path(node), time_cost, space_cost
        
        explored.add(node.state)
        
        for action in range(environment.action_space.n):
            child = Node(
                environment.sample(node.state, action),
                node,
                node.pathcost + 1,
                #l1_norm / l2_norm / chebyshev
                node.pathcost + 1 + Heu.l1_norm(
                    environment.state_to_pos(environment.sample(node.state, action)),
                    goalpos))  
            time_cost += 1
            
            if child.state not in queue and child.state not in explored:
                queue.add(child)
            elif present_with_higher_value(queue, child):
                queue.replace(child)
                
        space_cost = max(space_cost, len(queue) + len(explored))

### Check the results

In [13]:
def astar(environment, search_type):
    path, time_cost, space_cost = search_type(environment)
    return path, time_cost, space_cost

In [14]:
envname = "SmallMaze-v0"
environment = gym.make(envname)

solution_ts, time_ts, memory_ts = astar(environment, astar_tree_search)
solution_gs, time_gs, memory_gs = astar(environment, astar_graph_search)
heuristic = str(input('Which heuristic did you use? (l1_norm, l2_norm, chebyshev)\n'))

results = CheckResult_L2A2([solution_ts, time_ts, memory_ts], [solution_gs, time_gs, memory_gs], heuristic, env)
results.check_sol_ts()
results.check_sol_gs()

Which heuristic did you use? (l1_norm, l2_norm, chebyshev)
 l1_norm


#########################################
#######  A* TREE SEARCH PROBLEM  ########
#########################################
Your solution: [(np.int64(0), np.int64(1)), (np.int64(1), np.int64(1)), (np.int64(2), np.int64(1)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 8361
Max n° of nodes in memory: 6271

===> Your solution is correct!

##########################################
#######  A* GRAPH SEARCH PROBLEM  ########
##########################################
Your solution: [(np.int64(0), np.int64(1)), (np.int64(1), np.int64(1)), (np.int64(2), np.int64(1)), (np.int64(2), np.int64(0)), (np.int64(3), np.int64(0)), (np.int64(4), np.int64(0)), (np.int64(4), np.int64(1)), (np.int64(4), np.int64(2)), (np.int64(4), np.int64(3))]
N° of nodes explored: 61
Max n° of nodes in memory: 16

===> Your solution is correct!

